In [2]:
import os
import json
import shutil
import numpy as np
from collections import Counter


def stratified_coco_sampling(annotation_path, image_dir, label_dir, n_samples, output_dir):
    """
    COCO 전체 클래스 비율을 반영하여 n개의 이미지를 샘플링하고 
    해당 이미지/라벨을 output_dir에 저장하는 함수.
    """

    print("=== COCO Annotation Load ===")
    with open(annotation_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = coco["images"]
    annotations = coco["annotations"]

    # -----------------------------
    # 1. 전체 클래스 비율 계산
    # -----------------------------
    print("=== 전체 클래스 비율 계산 ===")
    class_counts = Counter([ann["category_id"] for ann in annotations])
    total_objects = sum(class_counts.values())
    class_ratios = {cid: class_counts[cid] / total_objects for cid in class_counts}

    print("전체 클래스 비율:")
    for cid, r in class_ratios.items():
        print(f" - Class {cid}: {r:.4f}")

    # -----------------------------
    # 2. 이미지별 포함 클래스 계산
    # -----------------------------
    print("\n=== 이미지별 클래스 수집 ===")
    img_classes = {img["id"]: [] for img in images}

    for ann in annotations:
        img_classes[ann["image_id"]].append(ann["category_id"])    # 이미지별 포함 클래스 목록

    # 클래스 → 포함 이미지 목록
    class_to_images = {cid: [] for cid in class_counts.keys()}
    for img_id, cls_list in img_classes.items():
        for cid in set(cls_list):  # unique class
            class_to_images[cid].append(img_id)

    # -----------------------------
    # 3. 클래스 비율에 맞춰 샘플 개수 계산
    # -----------------------------
    print("\n=== 클래스 비율 기반 샘플링 개수 ===")
    class_sample_target = {
        cid: max(1, int(ratio * n_samples)) for cid, ratio in class_ratios.items()
    }

    for cid, num in class_sample_target.items():
        print(f" - Class {cid}: {num} images target")

    # -----------------------------
    # 4. 클래스별로 이미지 샘플링
    # -----------------------------
    print("\n=== Stratified Image Sampling ===")
    selected_images = set()

    for cid, target in class_sample_target.items():
        candidates = class_to_images[cid]

        if len(candidates) == 0:
            continue

        if len(candidates) <= target:
            sampled = candidates
        else:
            sampled = list(np.random.choice(candidates, target, replace=False))

        selected_images.update(sampled)

    # 샘플 개수를 n_samples로 맞추기 (초과 시 랜덤 제거)
    selected_images = list(selected_images)

    if len(selected_images) > n_samples:
        selected_images = list(np.random.choice(selected_images, n_samples, replace=False))

    print(f"\n최종 선택된 이미지 수: {len(selected_images)}")

    # -----------------------------
    # 5. 출력 디렉토리 생성
    # -----------------------------
    print("\n=== 파일 복사 ===")
    if os.path.isdir(output_dir):
        shutil.rmtree(output_dir)

    out_img_dir = os.path.join(output_dir, "images")
    out_lbl_dir = os.path.join(output_dir, "labels")
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    # COCO image_id → file_name 매핑
    id_to_filename = {img["id"]: img["file_name"] for img in images}

    for img_id in selected_images:
        fname = id_to_filename[img_id]

        # 이미지 복사
        shutil.copy2(os.path.join(image_dir, fname),
                     os.path.join(out_img_dir, fname))

        # 라벨 파일명 
        label_file = os.path.splitext(fname)[0] + ".json"
        if os.path.exists(os.path.join(label_dir, label_file)):
            shutil.copy2(os.path.join(label_dir, label_file),
                         os.path.join(out_lbl_dir, label_file))

    print("\n=== 완료! ===")
    print(f"샘플링된 데이터는 '{output_dir}' 에 저장되었습니다.")


stratified_coco_sampling(
    annotation_path="/home/elicer/dev/jh/data/train.json",
    image_dir="/home/elicer/dev/detectron2/final_data/train/images",
    label_dir="/home/elicer/dev/detectron2/final_data/train/labels",
    n_samples=int(len(os.listdir("/home/elicer/dev/detectron2/final_data/train/images"))*0.3),
    output_dir="/home/elicer/dev/jh/data/train_subset"
)

=== COCO Annotation Load ===
=== 전체 클래스 비율 계산 ===
전체 클래스 비율:
 - Class 1: 0.1045
 - Class 2: 0.0580
 - Class 3: 0.2597
 - Class 4: 0.0386
 - Class 5: 0.2946
 - Class 6: 0.0878
 - Class 7: 0.0467
 - Class 8: 0.0001
 - Class 9: 0.0000
 - Class 10: 0.0481
 - Class 11: 0.0032
 - Class 12: 0.0362
 - Class 13: 0.0005
 - Class 14: 0.0006
 - Class 15: 0.0051
 - Class 16: 0.0033
 - Class 17: 0.0048
 - Class 18: 0.0010
 - Class 19: 0.0001
 - Class 20: 0.0022
 - Class 21: 0.0006
 - Class 22: 0.0001
 - Class 23: 0.0004
 - Class 24: 0.0004
 - Class 25: 0.0005
 - Class 26: 0.0003
 - Class 27: 0.0007
 - Class 28: 0.0018
 - Class 29: 0.0000
 - Class 30: 0.0000
 - Class 31: 0.0000

=== 이미지별 클래스 수집 ===

=== 클래스 비율 기반 샘플링 개수 ===
 - Class 1: 1625 images target
 - Class 2: 902 images target
 - Class 3: 4041 images target
 - Class 4: 600 images target
 - Class 5: 4584 images target
 - Class 6: 1366 images target
 - Class 7: 726 images target
 - Class 8: 1 images target
 - Class 9: 1 images target
 - Class 10

In [1]:
# val 데이터셋은 random하게

# 랜덤으로 이미지 추출하여 train, val set 생성
import os
import shutil
import numpy as np

def create_subset(image_dir, label_dir, n, output_dir):
    """
    랜덤으로 n개의 이미지와 라벨 세트를 뽑아 새로운 디렉토리에 저장하는 함수.

    Args:
        image_dir (str): 원본 이미지 디렉토리 경로.
        label_dir (str): 원본 라벨 디렉토리 경로.
        n (int): 뽑을 이미지-라벨 세트의 개수.
        output_dir (str): 출력 디렉토리 경로.

    Returns:
        None
    """
    # 이미지와 라벨 파일 리스트 가져오기
    image_files = sorted(os.listdir(image_dir))
    label_files = sorted(os.listdir(label_dir))

    print(image_files)

    # 랜덤 인덱스를 이용하여 n개의 세트 뽑기
    if len(image_files) < n:
        raise ValueError("The number of requested samples is greater than the available matched files.")

    indices = np.random.randint(0, len(image_files), n)

    print(len(indices))
    print(indices)

    # 출력 디렉토리 생성 (기존 디렉토리 있으면 삭제 후 재생성)
    print(output_dir)
    if os.path.isdir(output_dir):
        shutil.rmtree(output_dir)
    else:
        output_image_dir = os.path.join(output_dir, 'images')
        output_label_dir = os.path.join(output_dir, 'labels')
        os.makedirs(output_image_dir, exist_ok=True)
        os.makedirs(output_label_dir, exist_ok=True)

    # 파일 복사
    for index in indices:
        shutil.copy2(os.path.join(image_dir, image_files[index]), os.path.join(output_image_dir, image_files[index]))
        shutil.copy2(os.path.join(label_dir, label_files[index]), os.path.join(output_label_dir, label_files[index]))

In [2]:
# val n장 추출
image_dir = '/home/elicer/dev/detectron2/final_data/val/images'
label_dir = '/home/elicer/dev/detectron2/final_data/val/labels'

n = 30  # 뽑을 세트 수
output_dir = '/home/elicer/dev/jh/data/val_subset'

create_subset(image_dir, label_dir, n, output_dir)

['08_174514_221206_05.jpg', '08_174514_221206_07.jpg', '08_174514_221206_08.jpg', '08_174514_221206_33.jpg', '08_174514_221206_35.jpg', '08_174514_221206_36.jpg', '08_174514_221206_40.jpg', '08_174534_221206_01.jpg', '08_174534_221206_02.jpg', '08_174534_221206_13.jpg', '08_174534_221206_15.jpg', '08_174534_221206_23.jpg', '08_174534_221206_31.jpg', '08_174554_221206_01.jpg', '08_174554_221206_07.jpg', '08_174554_221206_16.jpg', '08_174554_221206_19.jpg', '08_174554_221206_24.jpg', '08_174554_221206_29.jpg', '08_174554_221206_30.jpg', '08_174554_221206_31.jpg', '08_174554_221206_35.jpg', '08_174614_221206_05.jpg', '08_174614_221206_21.jpg', '08_174614_221206_23.jpg', '08_174614_221206_25.jpg', '08_174614_221206_28.jpg', '08_174614_221206_33.jpg', '08_174634_221206_08.jpg', '08_174634_221206_14.jpg', '08_174634_221206_17.jpg', '08_174634_221206_23.jpg', '08_174634_221206_27.jpg', '08_174634_221206_36.jpg', '08_174634_221206_40.jpg', '08_174654_221206_11.jpg', '08_174654_221206_13.jpg', 

In [4]:
# 라벨 통합하는 코드
import json
import cv2
import numpy as np

def calculate_area(polygon):
    x = np.array(polygon[::2])
    y = np.array(polygon[1::2])
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def calculate_bbox(polygon):
    x = polygon[::2]
    y = polygon[1::2]
    return [min(x), min(y), max(x) - min(x), max(y) - min(y)]

def convert_to_coco(input_dir, output_file, directory):
    # 초기 세팅
    coco = {
        "images": [],
        "annotations": [],
        "categories": []
    }

    annotation_id = 0
    category_id_map = {}
    category_id_counter = 1

    print(directory, len(os.listdir(input_dir)))

    # 라벨 변경
    for filename in os.listdir(input_dir):
        if filename.endswith('.json'):
            with open(os.path.join(input_dir, filename), 'r') as f:
                data = json.load(f)

                # 이미지 정보
                img_filename = filename.replace('.json', '.jpg')

                img = cv2.imread('/home/elicer/dev/jh/data/' + directory + '/images/' + img_filename)
                height, width, _ = img.shape

                image_info = {
                    "id": len(coco["images"]),
                    "file_name": img_filename, # 이미지와 라벨의 파일명은 같음
                    "width": width,
                    "height": height
                }
                coco["images"].append(image_info)

                '''
                # 2D bbox 어노테이션은 필요하면 추가
                for bbox2d in data.get("bbox2d", []):
                    category_name = bbox2d["name"]
                    if category_name not in category_id_map:
                        category_id_map[category_name] = category_id_counter
                        coco["categories"].append({
                            "id": category_id_counter,
                            "name": category_name
                        })
                        category_id_counter += 1

                    bbox = bbox2d["bbox"]
                    x_min, y_min, x_max, y_max = bbox
                    width = x_max - x_min
                    height = y_max - y_min

                    annotation = {
                        "id": annotation_id,
                        "image_id": image_info["id"],
                        "category_id": category_id_map[category_name],
                        "bbox": [x_min, y_min, width, height],
                        "area": width * height,
                        "iscrowd": 0
                    }
                    coco["annotations"].append(annotation)
                    annotation_id += 1
                '''

                # Add segmentations
                for segmentation in data.get("annotations", []):
                    category_name = segmentation["class"]
                    if category_name not in category_id_map:
                        category_id_map[category_name] = category_id_counter
                        coco["categories"].append({
                            "id": category_id_counter,
                            "name": category_name
                        })
                        category_id_counter += 1

                    # segmentation을 [[x1, y1], [x1, y1], ...] => [x1, y1, x1, y1, ...] 형식으로 수정
                    # new_seg = []
                    # for x1, y1 in segmentation['polygon']:
                    #     new_seg.append(x1)
                    #     new_seg.append(y1)

                    new_seg = segmentation['polygon']  # 이미 [x1, y1, x2, y2, ...] 형태이므로 변환없이 진행


                    # 면적 및 bbox 계산
                    area = calculate_area(new_seg)
                    bbox = calculate_bbox(new_seg)

                    annotation = {
                        "id": annotation_id,
                        "image_id": image_info["id"],
                        "category_id": category_id_map[category_name],
                        "segmentation": [new_seg],
                        "area": area,
                        "bbox": bbox,
                        "iscrowd": 0
                    }
                    coco["annotations"].append(annotation)
                    annotation_id += 1

    # Save the result to a JSON file
    with open(output_file, 'w') as f: # encoding='utf-8' 조건 추가 가능하지만 오래걸림
        json.dump(coco, f, indent=4) # ensure_ascii=False 조건을 추가하여 한글 깨짐을 해결할 수 있으나 시간 오래 걸림

In [5]:
# .json 파일 생성
for d in ('train_subset', 'val_subset'):
    print(f'{d} start')
    input_dir = '/home/elicer/dev/jh/data/' + d + '/labels'
    output_file = '/home/elicer/dev/jh/data/' + d + '.json'

    convert_to_coco(input_dir, output_file, d)

train_subset start
train_subset 13708
val_subset start
val_subset 30
